In [1]:
from pathlib import Path
import pdfplumber
import re
import shutil
import json

In [2]:
# Base project data folder
BASE_DATA_DIR = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA"
)

RAW_DOCS_DIR = BASE_DATA_DIR
EXTRACTED_DIR = BASE_DATA_DIR / "Extracted"

# Ensure folders exist
RAW_DOCS_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Raw docs dir:   {RAW_DOCS_DIR}")
print(f"📂 Extracted dir:  {EXTRACTED_DIR}")

📂 Raw docs dir:   C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA
📂 Extracted dir:  C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\Extracted


In [3]:
def clean_text(text: str, is_table: bool = False) -> str:
    if not text:
        return ""
    text = text.replace("\x00", "")
    text = re.sub(r"\r\n", "\n", text)
    if is_table:
        text = re.sub(r"\n{4,}", "\n\n\n", text)
        text = re.sub(r"[ ]{3,}", "  ", text)
    else:
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

In [4]:
def extract_page_content(page) -> str:
    """
    Extracts both text and tables from a pdfplumber page.
    Tables are tagged with [TABLE]...[/TABLE] markers for downstream chunking.
    """
    parts = []

    # --- Extract tables ---
    try:
        tables = page.extract_tables()
        for table in tables:
            rows = []
            for row in table:
                cleaned = [cell.strip() if cell else "" for cell in row]
                rows.append(" | ".join(cleaned))
            table_text = "\n".join(rows)
            if table_text.strip():
                parts.append(f"[TABLE]\n{clean_text(table_text, is_table=True)}\n[/TABLE]")
    except Exception as e:
        print(f"   ⚠️  Table extraction error on page: {e}")

    # --- Extract regular text ---
    try:
        text = page.extract_text()
        if text:
            parts.append(clean_text(text))
    except Exception as e:
        print(f"   ⚠️  Text extraction error on page: {e}")

    return "\n\n".join(parts)

In [5]:
def convert_pdf_to_txt(pdf_path: Path, output_folder: Path) -> None:
    txt_path = output_folder / f"{pdf_path.stem}.txt"

    if txt_path.exists():
        print(f"⏭️  SKIPPED (already exists): {txt_path.name}")
        return

    try:
        pages_content = []
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages, start=1):
                content = extract_page_content(page)
                if content:
                    pages_content.append(f"[PAGE {i}]\n{content}")

        extracted_count = len(pages_content)

        if extracted_count == 0:
            print(f"⚠️  SCANNED PDF (no extractable text): {pdf_path.name}")
            return
        elif extracted_count < total_pages * 0.5:
            print(f"⚠️  PARTIAL EXTRACTION ({extracted_count}/{total_pages} pages): {pdf_path.name}")

        # ✅ Add metadata header so chunker knows this is a text doc
        header = (
            f"[META]\n"
            f"chunk_type: text\n"
            f"source_file: {pdf_path.name}\n"
            f"category: {pdf_path.parent.name}\n"
            f"total_pages: {total_pages}\n"
            f"[/META]\n\n"
        )

        full_text = header + "\n\n".join(pages_content)
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"✅ CREATED TXT ({extracted_count}/{total_pages} pages): {txt_path.name}")

    except Exception as e:
        print(f"❌ FAILED: {pdf_path.name} → {e}")

In [6]:
def validate_and_copy_json(json_file: Path, dest_file: Path) -> None:
    if dest_file.exists():
        print(f"⏭️  SKIPPED (already exists): {json_file.name}")
        return

    try:
        with open(json_file, encoding="utf-8") as f:
            data = json.load(f)

        # Stamp it so downstream knows it's a schema, not a text document
        if isinstance(data, dict):
            data["_meta"] = {
                "chunk_type": "schema",
                "source_file": json_file.name,
                "category": json_file.parent.name
            }
        elif isinstance(data, list):
            for item in data:
                if isinstance(item, dict):
                    item["_meta"] = {
                        "chunk_type": "schema",
                        "source_file": json_file.name,
                        "category": json_file.parent.name
                    }

        dest_file.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"📋 SCHEMA COPIED + ANNOTATED: {json_file.name}")

    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON schema {json_file.name}: {e}")
    except Exception as e:
        print(f"❌ Error copying {json_file.name}: {e}")

In [7]:
def convert_all_documents(
    raw_root: Path = RAW_DOCS_DIR,
    extracted_root: Path = EXTRACTED_DIR
) -> None:
    """
    Iterates through all category folders and processes PDFs and JSONs.
    PDFs are converted to TXT with page markers and table tags.
    JSONs are validated and copied as-is.
    """
    if not raw_root.exists():
        print(f"❌ Raw documents folder does not exist: {raw_root}")
        return

    for category_dir in raw_root.iterdir():
        if not category_dir.is_dir():
            continue
        if category_dir.name.lower() == extracted_root.name.lower():
            continue

        output_category_dir = extracted_root / category_dir.name
        output_category_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n📁 Processing category: {category_dir.name}")

        # --- PDFs ---
        pdf_files = list(category_dir.glob("*.pdf"))
        if not pdf_files:
            print("   (No PDFs found in this category)")
        else:
            for pdf_file in pdf_files:
                convert_pdf_to_txt(pdf_file, output_category_dir)

        # --- JSONs ---
        json_files = list(category_dir.glob("*.json"))
        if not json_files:
            print("   (No JSON files found in this category)")
        else:
            for json_file in json_files:
                dest_file = output_category_dir / json_file.name
                validate_and_copy_json(json_file, dest_file)

In [ ]:
convert_all_documents()


📁 Processing category: Cleaned_Generative
   (No PDFs found in this category)
   (No JSON files found in this category)

📁 Processing category: Database Tables
   (No PDFs found in this category)
📋 SCHEMA COPIED + ANNOTATED: ACT_PAR.JSON
📋 SCHEMA COPIED + ANNOTATED: ART_PAR.JSON
📋 SCHEMA COPIED + ANNOTATED: CHG_DAT.json
📋 SCHEMA COPIED + ANNOTATED: CHL_DAT .json
📋 SCHEMA COPIED + ANNOTATED: CHL_DAT.json
📋 SCHEMA COPIED + ANNOTATED: MIE_DAT.json
📋 SCHEMA COPIED + ANNOTATED: MIL_DAT.json
📋 SCHEMA COPIED + ANNOTATED: MVT_DAT.json
📋 SCHEMA COPIED + ANNOTATED: OPE_DAT.json
📋 SCHEMA COPIED + ANNOTATED: OPL_DAT.json
📋 SCHEMA COPIED + ANNOTATED: QUA_PAR.json
📋 SCHEMA COPIED + ANNOTATED: REA_DAT.json
📋 SCHEMA COPIED + ANNOTATED: REE_DAT.json
📋 SCHEMA COPIED + ANNOTATED: REL_DAT.json
📋 SCHEMA COPIED + ANNOTATED: SEX_DAT.json
📋 SCHEMA COPIED + ANNOTATED: stk_dat.json
📋 SCHEMA COPIED + ANNOTATED: TIE_PAR.JSON
📋 SCHEMA COPIED + ANNOTATED: ZEM_DAT.json

📁 Processing category: GENERAL
✅ CREATED TXT 

: 